In [ ]:
### HELPER FUNCTIONS

# @param:
#   - scale_path: path to scalefactors_json.json
# @return: 
#   - spot_px --> side length in pixels of one bin. 
#                 Depends on bin size, always in relation to the original image (Visium HD: .btf file) 
def get_binsize(scale_path):
    """
    Loads 'spot_diameter_fullres' from scalefactors_json.json.
    For Visium HD, this corresponds to the side length of a 2 µm bin in full-resolution pixels.
    """
    with open(scale_path, "r") as f:
        scales = json.load(f)

    # In full-resolution BTF coordinates, this is the correct field.
    spot_px = scales["spot_diameter_fullres"]
    return spot_px



# Assigns bins to image patches based on coordinates. Assumes crs=None (i.e. pixels).
# @param:
#   - bin_df: dataframe containg bin information, such as coordinates and clustering 
#   - patch_df: dataframe containg patch information, such as coordinates and clustering 
#   - bin_cols: list of column names to keep for merging from bins.
#   - patch_cols: list of column names to keep for mergin from patches. 
#   - id: string id column from bins (usually "barcode").
#   - scalefactor_path: string path to scalefactors_json.json. JSON should contain bin side length in pixels.
#   - output_dir: string directory to store resulting dataframe.
#   - box_size: integer patch size in pixels. For UNI model usually 224x224. 
#               NOTE, that both bins and patches are expected to be perfectly square.
# @return: 
#   - best_matches: dataframe. 1 bin = 1 row, with columns: 
#                   [bin_cols(...), patch_cols(...), patch_id, intersection_area, cell_area, percent_covered]
def assign_bins_to_patches(bin_df, patch_df, bin_cols, patch_cols, id, scalefactor_path, output_dir, box_size=224):
    if bin_df.shape[0] == 0 or patch_df.shape[0] == 0:
        print("WARNING: Empty bin or patch info does not contain any content. ")
        return {}
    # if (ground_truth[['pxl_col_in_fullres', 'pxl_row_in_fullres']].isna()):
        # print("WARNING: Empty entries in ground truth. Exiting. ")
        # return {}

    
    # 1. Turn patches into box polygons
    patch_geometries = [
        box(row['X'], row['Y'], row['X'] + box_size, row['Y'] + box_size)
        for _, row in patch_df.iterrows()
    ]
    #patches_gdf = gpd.GeoDataFrame(patch_df[['Patch_ID', 'leiden_0.2', 'leiden_0.5', 'leiden_1.0', 'leiden_2.0']], geometry=patch_geometries, crs=None)
    patches_gdf = gpd.GeoDataFrame(patch_df[patch_cols], geometry=patch_geometries, crs=None)

    # 2. Turn bins into smaller box polygons
    scalefactor = get_binsize(scalefactor_path)
    print("scalefactor: ", scalefactor, type(scalefactor))
    bin_geometries = [
        box(row['pxl_col_in_fullres'], 
            row['pxl_row_in_fullres'], 
            row['pxl_col_in_fullres'] + scalefactor, 
            row['pxl_row_in_fullres'] + scalefactor)
        for _, row in bin_df.iterrows()
    ]
    #bins_gdf = gpd.GeoDataFrame(bin_df[['barcode', 'tissuetype', 'in_tissue']], geometry=bin_geometries, crs=None)
    bins_gdf = gpd.GeoDataFrame(bin_df[bin_cols], geometry=bin_geometries, crs=None)

    print(f"Patch bounds:", bins_gdf.total_bounds)  # minx, miny, maxx, maxy of nuclei
    print(f"Bin bounds:", patches_gdf.total_bounds)

    # 3. Spatial join: assign each bin to a patch it falls within ---
    overlaps = gpd.overlay(bins_gdf, patches_gdf, how='intersection')

    # 4. Compute overlap and total bin area
    overlaps['intersection_area'] = overlaps.geometry.area
    overlaps['cell_area'] = overlaps.groupby(id)['intersection_area'].transform('sum')

    overlaps['percent_covered'] = overlaps['intersection_area'] / overlaps['cell_area']

    # 5. Keep only the patch covering the largest fraction
    best_matches = (
        overlaps.sort_values('percent_covered', ascending=False)
        .drop_duplicates(subset='barcode')
        .copy()
    )

    del overlaps, bins_gdf, patches_gdf, patch_geometries
    gc.collect()

    best_matches = best_matches.drop(["geometry"], axis=1)
    best_matches = best_matches.set_index(id)
    best_matches.to_parquet(os.path.join(output_dir, "bin_patch_clusters.parquet"))
    return best_matches



# Plots total UMI distribution before and after setting UMI threshold cutoff
# @param: 
#   - adata_: AnnData File where we get UMI counts from
#   - cut_off: cut off value. User determined.  
def total_umi_old(adata_, cut_off):
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))

    # Box plot
    axs[0].boxplot(adata_.obs["total_counts"], vert=False, widths=0.7, patch_artist=True, boxprops=dict(facecolor='skyblue'))
    axs[0].set_title('Total Counts')

    # Box plot after filtering
    axs[1].boxplot(adata_.obs["total_counts"][adata_.obs["total_counts"] > cut_off], vert=False, widths=0.7, patch_artist=True, boxprops=dict(facecolor='skyblue'))
    axs[1].set_title('Total Counts > ' + str(cut_off))

    # Remove y-axis ticks and labels
    for ax in axs:
        ax.get_yaxis().set_visible(False)

    plt.tight_layout()
    plt.show()
    

def total_umi(adata_, cut_off, plot_cap=None):
    import numpy as np
    import matplotlib.pyplot as plt

    x = adata_.obs["total_counts"].values
    x_filt = x[x > cut_off]

    # --- clip for visualization only ---
    if plot_cap is not None:
        x_plot = np.clip(x, None, plot_cap)
        x_filt_plot = np.clip(x_filt, None, plot_cap)
    else:
        x_plot = x
        x_filt_plot = x_filt

    fig, axs = plt.subplots(1, 2, figsize=(12, 4))

    axs[0].boxplot(x_plot, vert=False)
    axs[0].set_title("Total Counts")

    axs[1].boxplot(x_filt_plot, vert=False)
    axs[1].set_title(f"Total Counts > {cut_off}")

    for ax in axs:
        ax.get_yaxis().set_visible(False)

    plt.tight_layout()
    plt.show()



# Plots all bins based on their original pixel position, colored based on clustering results.
# @param:
#   - df: dataframe containing all bin information 
#   - x_col: column name for x coordinates in pixels
#   - y_col: column name for y coordinates in pixels
#   - cluster_col: column name for clustering column
def plot_bin_clusters(df, x_col, y_col, cluster_col, title=None):
    # Extract columns
    x = df[x_col]
    y = df[y_col]
    clusters = df[cluster_col].astype('category')  # ensure categorical

    # Convert clusters to integer codes for coloring
    codes = clusters.cat.codes
    labels = clusters.cat.categories

    # Create a discrete colormap with enough colors
    cmap = plt.get_cmap('tab20', len(labels))  # tab20 has up to 20 distinct colors

    # Plot
    plt.figure(figsize=(8, 8))
    scatter = plt.scatter(
        x,
        y,
        c=codes,
        cmap=cmap,
        s=0.01,
        alpha=0.8
    )

    # Reverse y-axis (image coordinates often start top-left)
    plt.gca().invert_yaxis()

    # Add colorbar with cluster labels
    cbar = plt.colorbar(scatter, ticks=range(len(labels)))
    cbar.ax.set_yticklabels(labels)

    # Labels
    if title is not None:
        plt.title(title)
    else:
        plt.title("No title")
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.show()


# Plots two scatter plots next to each other for comparison. For each scatter plot:
# Plots all bins based on their original pixel position, colored based on clustering results.
# @param:
#   - df: dataframe containing all bin information 
#   - x_col: column name for x coordinates in pixels
#   - y_col: column name for y coordinates in pixels
#   - cluster_col1: column name for clustering column for plot 1
#   - cluster_col2: column name for clustering column for plot 2
def plot_bin_clusters_side_by_side(df, x_col, y_col, title1, title2, cluster_col1, cluster_col2="ground_truth"):
    x = df[x_col]
    y = df[y_col]
    
    # Use categorical codes
    clusters_pred = df[cluster_col1].astype("category")
    clusters_true = df[cluster_col2].astype("category")

    panels = [(title1, clusters_pred),
              (title2, clusters_true)]

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, (title, cat_clusters) in zip(axes, panels):

        codes = cat_clusters.cat.codes
        labels = cat_clusters.cat.categories

        # Create a discrete colormap with enough colors
        cmap = plt.get_cmap('tab20', len(labels))  # discrete colormap

        scatter = ax.scatter(
            x, y,
            c=codes,
            cmap=cmap,
            s=0.01,
            alpha=0.8
        )

        ax.invert_yaxis()
        ax.set_title(title)
        ax.set_xlabel(x_col)
        ax.set_ylabel(y_col)

        # Colorbar with original cluster names
        cbar = fig.colorbar(scatter, ax=ax, ticks=range(len(labels)))
        cbar.ax.set_yticklabels(labels)

    plt.tight_layout()
    plt.show()




# Prints evaluation metrics: ARI, Normalised Mutual Info, Homogeneity, Completeness, V-Measure
# @param:
#   - df: dataframe containing bins with barcodes, true cluster column and predicted cluster column
#   - pred_col: predicted cluster column name
#   - truth_col: true cluster column name
def get_eval_scores(df, pred_col, truth_col):
    num_cluster_nans = df[pred_col].isna().sum()
    num_tissue_nans = df[truth_col].isna().sum()

    if num_cluster_nans > 0 or num_tissue_nans > 0:
        print(f"Found {num_cluster_nans} NaN values in '{pred_col}' and {num_tissue_nans} in '{truth_col}' out of {df.shape[0]} rows. Dropping rows ...")
        df = df.dropna(subset=[pred_col, truth_col])

    y_true = df[truth_col]
    y_pred = df[pred_col]

    results = {
        'Adjusted Rand Index': adjusted_rand_score(y_true, y_pred),
        'Normalized Mutual Info': normalized_mutual_info_score(y_true, y_pred),
        'Homogeneity': homogeneity_score(y_true, y_pred),
        'Completeness': completeness_score(y_true, y_pred),
        'V-Measure': v_measure_score(y_true, y_pred)
    }

    for k, v in results.items():
        print(f"{k}: {v:.3f}")


## Helper functions from CustomBinning.ipynb - aken from 10x genomics
# Plotting function for nuclei area distribution
def plot_nuclei_area(gdf,area_cut_off):
    fig, axs = plt.subplots(1, 2, figsize=(15, 4))
    # Plot the histograms
    axs[0].hist(gdf['area'], bins=50, edgecolor='black')
    axs[0].set_title('Nuclei Area')

    axs[1].hist(gdf[gdf['area'] < area_cut_off]['area'], bins=50, edgecolor='black')
    axs[1].set_title('Nuclei Area Filtered:'+str(area_cut_off))

    plt.tight_layout()
    plt.show()

## Clustering functions